In [18]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder,OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
data = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["TotalCharges"] = data["TotalCharges"].fillna(data["TotalCharges"].median())

In [4]:
data["Churn"] = data["Churn"].map({"Yes":1, "No":0})

In [5]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.7 MB/s eta 0:00:00


In [9]:
data.drop("customerID", axis=1, inplace=True)

In [19]:
cat_columns = data.select_dtypes(include=['object'])

print("Categorical Columns:")
print(cat_columns.columns)

data_one_hot_encoding = pd.get_dummies(
    data,
    columns=cat_columns.columns,
    drop_first=True,
    dtype=int
)

print("\nOne Hot Encoded Data:")
print(data_one_hot_encoding.head())

data_label_encoding = data.copy()

le = LabelEncoder()

for col in cat_columns.columns:
    data_label_encoding[col] = le.fit_transform(
        data_label_encoding[col].astype(str)
    )

print("\nLabel Encoded Data:")
print(data_label_encoding.head())

data_ordinal_encoding = data.copy()

oe = OrdinalEncoder()

data_ordinal_encoding[cat_columns.columns] = oe.fit_transform(
    data_ordinal_encoding[cat_columns.columns].astype(str)
)

print("\nOrdinal Encoded Data:")
print(data_ordinal_encoding.head())

import category_encoders as ce

data_binary_encoding = data.copy()

encoder = ce.BinaryEncoder(cols=['PaymentMethod'])
data_binary_encoding = encoder.fit_transform(data_binary_encoding)
print("\nBinary Encoded Data:")
print(data_binary_encoding.head())

Categorical Columns:
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'ServiceCount'],
      dtype='object')

One Hot Encoded Data:
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  Churn  \
0              0       1           29.85         29.85      0   
1              0      34           56.95       1889.50      0   
2              0       2           53.85        108.15      1   
3              0      45           42.30       1840.75      0   
4              0       2           70.70        151.65      1   

   AvgMonthlySpend  remaining_contract_months  ContractValue  gender_Male  \
0        14.925000                          1          29.85            0   
1        53.985714                          1          56.95            1   
2        36.050000          

In [10]:
data["AvgMonthlySpend"] = data["TotalCharges"] / (data["tenure"] + 1)

# Count subscribed services
service_cols = [
    "PhoneService",
    "Partner",
    "Dependents"
]

data["ServiceCount"] = data[service_cols].sum(axis=1)


In [11]:
data["remaining_contract_months"] = np.where(
    data.filter(like="Contract_One year").sum(axis=1)==1, 12,
    np.where(
        data.filter(like="Contract_Two year").sum(axis=1)==1, 24, 1
    )
)

data["ContractValue"] = data["MonthlyCharges"] * data["remaining_contract_months"]


In [12]:
X = data.drop("Churn", axis=1)
y = data["Churn"]


In [13]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

data_encoded = pd.get_dummies(data, drop_first=True)

X = data_encoded.drop("Churn", axis=1)
y = data_encoded["Churn"]

std_scaler = StandardScaler()
X_std = pd.DataFrame(std_scaler.fit_transform(X), columns=X.columns)

print("Standard Scaled:")
print(X_std.head())

mm_scaler = MinMaxScaler()
X_mm = pd.DataFrame(mm_scaler.fit_transform(X), columns=X.columns)

print("MinMax Scaled:")
print(X_mm.head())

Standard Scaled:
   SeniorCitizen    tenure  MonthlyCharges  TotalCharges  AvgMonthlySpend  \
0      -0.439916 -1.277445       -1.160323     -0.994242        -0.757979   
1      -0.439916  0.066327       -0.259629     -0.173244        -0.117801   
2      -0.439916 -1.236724       -0.362660     -0.959674        -0.411755   
3      -0.439916  0.514251       -0.746535     -0.194766        -0.346750   
4      -0.439916 -1.236724        0.197365     -0.940470        -0.174110   

   remaining_contract_months  ContractValue  gender_Male  Partner_Yes  \
0                        0.0      -1.160323    -1.009559     1.034530   
1                        0.0      -0.259629     0.990532    -0.966622   
2                        0.0      -0.362660     0.990532    -0.966622   
3                        0.0      -0.746535     0.990532    -0.966622   
4                        0.0       0.197365    -1.009559    -0.966622   

   Dependents_Yes  ...  PaymentMethod_Credit card (automatic)  \
0       -0.65401

In [22]:
# Convert target first
data["Churn"] = data["Churn"].map({"Yes":1, "No":0})

# Select numeric columns only
numeric_data = data.select_dtypes(include=['number'])

# Correlation
corr = numeric_data.corr()["Churn"].abs().sort_values(ascending=False)

print(corr.head(10))

print("Top Correlated Features:")
print(corr.head(10))

# ---------- Method 2: RFE ----------
model = LogisticRegression(max_iter=2000)

rfe = RFE(model, n_features_to_select=10)
rfe.fit(X_std, y)

selected_features = X.columns[rfe.support_]
print("RFE Selected Features:")
print(selected_features)

# ---------- Method 3: Tree Feature Importance ----------
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=X.columns)
print("Top Tree Importance Features:")
print(importance.sort_values(ascending=False).head(10))

# ---------- Method 4: Mutual Information ----------
mi = mutual_info_classif(X, y)
mi_scores = pd.Series(mi, index=X.columns)
print("Top Mutual Information Features:")
print(mi_scores.sort_values(ascending=False).head(10))

SeniorCitizen               NaN
tenure                      NaN
MonthlyCharges              NaN
TotalCharges                NaN
Churn                       NaN
AvgMonthlySpend             NaN
remaining_contract_months   NaN
ContractValue               NaN
Name: Churn, dtype: float64
Top Correlated Features:
SeniorCitizen               NaN
tenure                      NaN
MonthlyCharges              NaN
TotalCharges                NaN
Churn                       NaN
AvgMonthlySpend             NaN
remaining_contract_months   NaN
ContractValue               NaN
Name: Churn, dtype: float64
RFE Selected Features:
Index(['tenure', 'TotalCharges', 'AvgMonthlySpend',
       'InternetService_Fiber optic', 'OnlineSecurity_No internet service',
       'StreamingTV_Yes', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year'],
      dtype='object')
Top Tree Importance Features:
TotalCharges                      0.141747
tenure                 

In [23]:
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

print("After SMOTE:")
print(pd.Series(y_smote).value_counts())

# ---------- Random Undersampling ----------
rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X, y)

print("After Undersampling:")
print(pd.Series(y_under).value_counts())

After SMOTE:
Churn
0    5174
1    5174
Name: count, dtype: int64
After Undersampling:
Churn
0    1869
1    1869
Name: count, dtype: int64


In [24]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split temp into validation 15% and test 15%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train Shape :", X_train.shape)
print("Validation Shape :", X_val.shape)
print("Test Shape :", X_test.shape)

Train Shape : (4930, 40)
Validation Shape : (1056, 40)
Test Shape : (1057, 40)
